# U-Net++: seeds 123 and 2026

Previously supplied runner used for the new runs. Model from original cell 73, training pipeline adapted from cell 45; the original seed-42 U-Net++ training cell was absent. Seed-42 summary entries are references, not newly evaluated.

Source: `gd_unetpp_seeds123_2026.py`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB — ORIGINAL U-NET++ (BATCHNORM), SEEDS 123 AND 2026 ONLY

One Kaggle/Colab cell. No previous notebook variables or checkpoint required.
Architecture copied verbatim from the user's notebook computational benchmark:
UNetPlusPlusBN, base_channels=30, 8,051,791 parameters, six-channel input.
The shared controlled training/data/evaluation pipeline is copied from notebook
cell 45 and applied to that U-Net++ architecture. The original U-Net++ training
cell itself was not present in the supplied saved notebook.

70 maximum epochs, early stopping 12, batch 2 x accumulation 2, AdamW 1e-4,
weight decay 1e-4, ReduceLROnPlateau factor .5 / patience 4 / minimum 1e-6,
.35 BCE + .25 focal(gamma=2) + .40 Dice, train-only augmentation, no TTA.
Checkpoint selection uses the original coarse validation grid .20:.05:.85;
final threshold uses validation predictions only, grid .20:.01:.80.
SEED 42 IS NOT TRAINED. Its manuscript metrics are saved separately as a
reference, and never presented as a freshly evaluated run.

Each completed epoch saves resumable optimizer/scheduler/scaler/RNG state.
Each seed produces best checkpoint, final pooled test metrics, image-wise
metrics, validation threshold scan, training history, metadata and a ZIP.
"""
from __future__ import annotations

import csv
import gc
import hashlib
import json
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

RUN_SEEDS = [123, 2026]
DATASET_ROOT = None  # Automatic; first checks /kaggle/input/datasets/hemal7/gd-rgbms-db
if Path("/kaggle/working").exists():
    MASTER_OUT_DIR = Path("/kaggle/working/gd_unetpp_seeds123_2026")
elif Path("/content").exists():
    MASTER_OUT_DIR = Path("/content/gd_unetpp_seeds123_2026")
else:
    MASTER_OUT_DIR = Path.cwd() / "gd_unetpp_seeds123_2026"
MASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Existing seed-42 manuscript result, not a new training/evaluation run.
SEED42_REFERENCE = {
    "model": "U-Net++ (custom convolutional encoder, BatchNorm)",
    "seed": 42, "dice": .7845, "iou": .6454,
    "precision": .7765, "recall": .7926,
    "parameter_count": 8_051_791,
    "provenance": "existing manuscript seed-42 result; not rerun",
}

class UNetPPConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNetPlusPlusBN(nn.Module):

    def __init__(
        self,
        in_channels=6,
        base_channels=30
    ):
        super().__init__()

        c0 = base_channels
        c1 = base_channels * 2
        c2 = base_channels * 4
        c3 = base_channels * 8
        c4 = base_channels * 16

        self.pool = nn.MaxPool2d(2)

        # Encoder
        self.conv0_0 = UNetPPConvBlock(
            in_channels, c0
        )
        self.conv1_0 = UNetPPConvBlock(c0, c1)
        self.conv2_0 = UNetPPConvBlock(c1, c2)
        self.conv3_0 = UNetPPConvBlock(c2, c3)
        self.conv4_0 = UNetPPConvBlock(c3, c4)

        # Nested decoder
        self.conv0_1 = UNetPPConvBlock(
            c0 + c1, c0
        )
        self.conv1_1 = UNetPPConvBlock(
            c1 + c2, c1
        )
        self.conv2_1 = UNetPPConvBlock(
            c2 + c3, c2
        )
        self.conv3_1 = UNetPPConvBlock(
            c3 + c4, c3
        )

        self.conv0_2 = UNetPPConvBlock(
            c0 * 2 + c1, c0
        )
        self.conv1_2 = UNetPPConvBlock(
            c1 * 2 + c2, c1
        )
        self.conv2_2 = UNetPPConvBlock(
            c2 * 2 + c3, c2
        )

        self.conv0_3 = UNetPPConvBlock(
            c0 * 3 + c1, c0
        )
        self.conv1_3 = UNetPPConvBlock(
            c1 * 3 + c2, c1
        )

        self.conv0_4 = UNetPPConvBlock(
            c0 * 4 + c1, c0
        )

        self.segmentation_head = nn.Conv2d(
            c0,
            1,
            1
        )

    @staticmethod
    def upsample(x, reference):

        return F.interpolate(
            x,
            size=reference.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    def forward(self, x):

        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x2_0 = self.conv2_0(self.pool(x1_0))
        x3_0 = self.conv3_0(self.pool(x2_0))
        x4_0 = self.conv4_0(self.pool(x3_0))

        x0_1 = self.conv0_1(
            torch.cat([
                x0_0,
                self.upsample(x1_0, x0_0)
            ], dim=1)
        )

        x1_1 = self.conv1_1(
            torch.cat([
                x1_0,
                self.upsample(x2_0, x1_0)
            ], dim=1)
        )

        x2_1 = self.conv2_1(
            torch.cat([
                x2_0,
                self.upsample(x3_0, x2_0)
            ], dim=1)
        )

        x3_1 = self.conv3_1(
            torch.cat([
                x3_0,
                self.upsample(x4_0, x3_0)
            ], dim=1)
        )

        x0_2 = self.conv0_2(
            torch.cat([
                x0_0,
                x0_1,
                self.upsample(x1_1, x0_0)
            ], dim=1)
        )

        x1_2 = self.conv1_2(
            torch.cat([
                x1_0,
                x1_1,
                self.upsample(x2_1, x1_0)
            ], dim=1)
        )

        x2_2 = self.conv2_2(
            torch.cat([
                x2_0,
                x2_1,
                self.upsample(x3_1, x2_0)
            ], dim=1)
        )

        x0_3 = self.conv0_3(
            torch.cat([
                x0_0,
                x0_1,
                x0_2,
                self.upsample(x1_2, x0_0)
            ], dim=1)
        )

        x1_3 = self.conv1_3(
            torch.cat([
                x1_0,
                x1_1,
                x1_2,
                self.upsample(x2_2, x1_0)
            ], dim=1)
        )

        x0_4 = self.conv0_4(
            torch.cat([
                x0_0,
                x0_1,
                x0_2,
                x0_3,
                self.upsample(x1_3, x0_0)
            ], dim=1)
        )

        return self.segmentation_head(x0_4)


def run_one(run_seed):
    # =============================================================================
    # 1. CONFIGURATION
    # =============================================================================

    # Set manually only if automatic discovery fails.
    # BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"
    # BASE_DIR = "/content/GD-RGBMS-DB"
    BASE_DIR = DATASET_ROOT

    SEED = int(run_seed)
    TARGET_SIZE = (896, 672)  # width, height
    INPUT_CHANNEL_ORDER = ["R", "G", "B", "Green", "Red-Edge", "NIR"]

    BATCH_SIZE = 2
    ACCUMULATION_STEPS = 2
    MAX_EPOCHS = 70
    PATIENCE = 12
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0

    # Original notebook U-Net++: base=30, BatchNorm, 8,051,791 parameters.
    BASE_CHANNELS = 30

    # BatchNorm is used in the original U-Net++ convolution blocks.
    NORMALIZATION = "BatchNorm"

    # Float16 CPU RAM cache is around 2.4-2.6 GB for six-channel train+validation.
    CACHE_TRAIN_AND_VALIDATION = True
    USE_CHANNELS_LAST = True
    SHOW_DOWNLOAD_LINK = False

    MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

    # Train-split-only statistics; order matches INPUT_CHANNEL_ORDER.
    MEAN = np.array(
        [
            0.41819797,
            0.46573568,
            0.25167707,
            0.21822916,
            0.16206901,
            0.23585168,
        ],
        dtype=np.float32,
    )
    STD = np.array(
        [
            0.15404519,
            0.15691774,
            0.13375558,
            0.07147331,
            0.05124885,
            0.04640943,
        ],
        dtype=np.float32,
    )

    # Coarse thresholds during checkpoint selection; fine thresholds after loading
    # the best checkpoint. Both searches use validation predictions only.
    TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()
    FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

    if Path("/kaggle/working").exists():
        WORK_DIR = Path("/kaggle/working")
    elif Path("/content").exists():
        WORK_DIR = Path("/content")
    else:
        WORK_DIR = Path.cwd()

    OUT_DIR = MASTER_OUT_DIR / f"seed{SEED}"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH = OUT_DIR / f"best_unetpp_bn_6ch_seed{SEED}.pt"
    LAST_CHECKPOINT_PATH = OUT_DIR / "last_training_checkpoint.pt"
    configuration = {
        "seed": SEED, "model": "UNetPlusPlusBN", "base_channels": BASE_CHANNELS,
        "parameter_count": 8_051_791, "input_channels": INPUT_CHANNEL_ORDER,
        "input_size": list(TARGET_SIZE), "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS, "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": PATIENCE, "initial_lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "mean": MEAN.tolist(), "std": STD.tolist(),
        "training_thresholds": TRAIN_THRESHOLDS, "final_thresholds": FINAL_THRESHOLDS,
        "source_sha256": PROGRAM_SHA256,
    }
    configuration_path = OUT_DIR / "requested_configuration.json"
    if configuration_path.exists():
        previous = json.loads(configuration_path.read_text())
        if previous != configuration:
            raise RuntimeError(f"Different configuration already exists in {OUT_DIR}; use a new output folder.")
    else:
        configuration_path.write_text(json.dumps(configuration, indent=2))
    summary_path = OUT_DIR / "run_summary.json"
    if summary_path.exists():
        completed = json.loads(summary_path.read_text())
        if completed.get("source_sha256") != PROGRAM_SHA256:
            raise RuntimeError("Completed result does not match this runner's configuration.")
        print(f"Seed {SEED}: already completed; reusing saved actual test metrics.")
        return completed


    try:
        BILINEAR = Image.Resampling.BILINEAR
        NEAREST = Image.Resampling.NEAREST
    except AttributeError:
        BILINEAR = Image.BILINEAR
        NEAREST = Image.NEAREST


    # =============================================================================
    # 2. DATASET DISCOVERY AND REPRODUCIBILITY
    # =============================================================================

    def read_list(path: Path) -> list[str]:
        with open(path, "r", encoding="utf-8") as file:
            return [line.strip() for line in file if line.strip()]


    def valid_root(path: Path) -> bool:
        required = [
            path / "RGB",
            path / "MS",
            path / "Mark",
            path / "train_list.txt",
            path / "val_list.txt",
            path / "test_list.txt",
        ]
        if not all(item.exists() for item in required):
            return False
        return (
            len(read_list(path / "train_list.txt")) == 280
            and len(read_list(path / "val_list.txt")) == 60
            and len(read_list(path / "test_list.txt")) == 60
        )


    def find_dataset_root() -> Path:
        fast_candidates = [
            Path("/kaggle/input/datasets/hemal7/gd-rgbms-db"),
            Path("/content/GD-RGBMS-DB"),
            Path("/content/gd-rgbms-db"),
            Path("/content/drive/MyDrive/GD-RGBMS-DB"),
        ]
        for candidate in fast_candidates:
            if valid_root(candidate):
                return candidate
        search_roots = [
            Path("/kaggle/input"),
            Path("/content"),
            Path("/content/drive/MyDrive"),
        ]
        candidates: list[Path] = []
        for root in search_roots:
            if root.exists():
                for item in root.rglob("train_list.txt"):
                    if valid_root(item.parent):
                        candidates.append(item.parent)
        if not candidates:
            raise FileNotFoundError(
                "The exact 280/60/60 dataset root was not found. Set BASE_DIR "
                "manually near the top of this script."
            )
        candidates.sort(
            key=lambda path: ("gd-rgbms" not in str(path).lower(), len(str(path)))
        )
        return candidates[0]


    if BASE_DIR is None:
        BASE_DIR = find_dataset_root()
    BASE_DIR = Path(BASE_DIR)


    def seed_everything(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")


    seed_everything(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type != "cuda":
        raise RuntimeError("Enable a GPU accelerator in Kaggle/Colab, then run this cell again.")
    PIN_MEMORY = DEVICE.type == "cuda"

    train_names = read_list(BASE_DIR / "train_list.txt")
    val_names = read_list(BASE_DIR / "val_list.txt")
    test_names = read_list(BASE_DIR / "test_list.txt")

    split_sets = [set(train_names), set(val_names), set(test_names)]
    if (
        split_sets[0] & split_sets[1]
        or split_sets[0] & split_sets[2]
        or split_sets[1] & split_sets[2]
    ):
        raise ValueError("Filename overlap exists across the train/val/test lists.")

    RGB_DIR = BASE_DIR / "RGB"
    MS_DIR = BASE_DIR / "MS"
    MARK_DIR = BASE_DIR / "Mark"

    print("=" * 80)
    print(f"SIX-CHANNEL U-NET++ — SEED {SEED} — NO TTA")
    print("=" * 80)
    print("Dataset       :", BASE_DIR)
    print("Output        :", OUT_DIR)
    print("Device        :", DEVICE)
    if DEVICE.type == "cuda":
        print("GPU           :", torch.cuda.get_device_name(0))
    print("Split         :", len(train_names), len(val_names), len(test_names))
    print("Input size    :", TARGET_SIZE, "(width, height)")
    print("Channels      :", INPUT_CHANNEL_ORDER)
    print("Batch/accum   :", BATCH_SIZE, "/", ACCUMULATION_STEPS)
    print("Normalization :", NORMALIZATION)


    # =============================================================================
    # 3. PREPROCESSING, AUGMENTATION, AND LOADERS
    # =============================================================================

    def find_path(directory: Path, stem: str, extensions: list[str]) -> Path:
        for extension in extensions:
            path = directory / f"{stem}{extension}"
            if path.exists():
                return path
        raise FileNotFoundError(f"Could not find '{stem}' inside {directory}")


    def binary_mask(image: Image.Image) -> np.ndarray:
        array = np.asarray(image.convert("RGB"), dtype=np.int16)
        return (
            np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80
        ).astype(np.float32)


    def load_raw(name: str) -> tuple[np.ndarray, np.ndarray, str]:
        stem = Path(str(name).strip()).name
        if Path(stem).suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}:
            stem = stem[: -len(Path(stem).suffix)]
        rgb_path = find_path(RGB_DIR, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"])
        ms_path = find_path(MS_DIR, stem, [".PNG", ".png", ".JPG", ".jpg"])
        mark_path = find_path(MARK_DIR, stem, [".png", ".PNG", ".jpg", ".JPG"])

        with Image.open(rgb_path) as item:
            rgb_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
        with Image.open(ms_path) as item:
            ms_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
        with Image.open(mark_path) as item:
            mark_image = item.convert("RGB").resize(TARGET_SIZE, NEAREST)

        rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
        ms = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
        mask = binary_mask(mark_image)
        image = np.concatenate([rgb, ms], axis=2).astype(np.float32)
        return image, mask, stem


    RAM_CACHE: dict[str, tuple[np.ndarray, np.ndarray, str]] = {}


    def preload_cache(names: list[str]) -> None:
        if not CACHE_TRAIN_AND_VALIDATION:
            return
        for name in tqdm(names, desc="Preloading train+val RAM cache"):
            image, mask, stem = load_raw(name)
            RAM_CACHE[name] = (
                image.astype(np.float16),
                mask.astype(np.uint8),
                stem,
            )
        approximate_gb = sum(
            image.nbytes + mask.nbytes
            for image, mask, _ in RAM_CACHE.values()
        ) / (1024**3)
        print(f"RAM cache: {len(RAM_CACHE)} images, {approximate_gb:.2f} GB")


    preload_cache(train_names + val_names)


    class GDDataset(Dataset):
        def __init__(self, names: list[str], augment: bool = False) -> None:
            self.names = list(names)
            self.augment = augment

        def __len__(self) -> int:
            return len(self.names)

        def __getitem__(self, index: int):
            name = self.names[index]
            if name in RAM_CACHE:
                cached_image, cached_mask, stem = RAM_CACHE[name]
                image = cached_image.astype(np.float32)
                mask = cached_mask.astype(np.float32)
            else:
                image, mask, stem = load_raw(name)

            if self.augment:
                if random.random() < 0.5:
                    image = np.flip(image, axis=1).copy()
                    mask = np.flip(mask, axis=1).copy()
                if random.random() < 0.5:
                    image = np.flip(image, axis=0).copy()
                    mask = np.flip(mask, axis=0).copy()
                if random.random() < 0.75:
                    gain = np.random.uniform(
                        0.88, 1.12, size=(1, 1, 3)
                    ).astype(np.float32)
                    bias = np.random.uniform(
                        -0.035, 0.035, size=(1, 1, 3)
                    ).astype(np.float32)
                    image[..., :3] = np.clip(
                        image[..., :3] * gain + bias, 0.0, 1.0
                    )

            image = ((image - MEAN) / STD).astype(np.float32)
            return (
                torch.from_numpy(image.transpose(2, 0, 1)).float(),
                torch.from_numpy(mask[None]).float(),
                stem,
            )


    train_loader = DataLoader(
        GDDataset(train_names, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    val_loader = DataLoader(
        GDDataset(val_names, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        GDDataset(test_names, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


    # =============================================================================
    # 4. ORIGINAL U-NET++ WITH BATCH NORMALIZATION
    # =============================================================================

    model = UNetPlusPlusBN(in_channels=6, base_channels=BASE_CHANNELS).to(DEVICE)

    if USE_CHANNELS_LAST and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    PARAMETER_COUNT = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    EXPECTED_PARAMETERS = 8_051_791
    if PARAMETER_COUNT != EXPECTED_PARAMETERS:
        raise RuntimeError(
            f"Architecture mismatch: expected {EXPECTED_PARAMETERS:,} parameters, "
            f"found {PARAMETER_COUNT:,}."
        )
    print("Trainable parameters:", f"{PARAMETER_COUNT:,} ({PARAMETER_COUNT / 1e6:.3f} M)")


    # =============================================================================
    # 5. LOSS AND METRICS
    # =============================================================================

    class ComboLoss(nn.Module):
        def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
            targets = targets.float()
            bce_map = F.binary_cross_entropy_with_logits(
                logits, targets, reduction="none"
            )
            bce = bce_map.mean()
            probabilities = torch.sigmoid(logits)
            pt = probabilities * targets + (1 - probabilities) * (1 - targets)
            focal = (bce_map * ((1 - pt) ** 2.0)).mean()
            dimensions = tuple(range(1, logits.ndim))
            intersection = (probabilities * targets).sum(dimensions)
            denominator = probabilities.sum(dimensions) + targets.sum(dimensions)
            dice_loss = 1 - (
                (2 * intersection + 1e-6) / (denominator + 1e-6)
            ).mean()
            return 0.35 * bce + 0.25 * focal + 0.40 * dice_loss


    criterion = ComboLoss()


    def confusion(probabilities, masks, threshold: float) -> list[int]:
        prediction = probabilities >= threshold
        target = masks >= 0.5
        return [
            int((prediction & target).sum().item()),
            int((prediction & ~target).sum().item()),
            int((~prediction & target).sum().item()),
            int((~prediction & ~target).sum().item()),
        ]


    def metrics(values: list[int]) -> dict[str, float]:
        tp, fp, fn, tn = values
        epsilon = 1e-7
        return {
            "dice": (2 * tp + epsilon) / (2 * tp + fp + fn + epsilon),
            "iou": (tp + epsilon) / (tp + fp + fn + epsilon),
            "precision": (tp + epsilon) / (tp + fp + epsilon),
            "recall": (tp + epsilon) / (tp + fn + epsilon),
            "accuracy": (tp + tn + epsilon) / (tp + fp + fn + tn + epsilon),
        }


    def add_values(left: list[int], right: list[int]) -> list[int]:
        return [old + new for old, new in zip(left, right)]


    # =============================================================================
    # 6. TRAINING AND VALIDATION-ONLY CHECKPOINT SELECTION
    # =============================================================================

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    except (TypeError, AttributeError):
        scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


    def move_images(images: torch.Tensor) -> torch.Tensor:
        images = images.to(DEVICE, non_blocking=True)
        if USE_CHANNELS_LAST and DEVICE.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)
        return images


    def autocast_context():
        try:
            return torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda")
        except AttributeError:
            return torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda")


    def train_epoch() -> float:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0

        for step, (images, masks, _) in enumerate(
            tqdm(train_loader, desc="Training", leave=False), start=1
        ):
            images = move_images(images)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast_context():
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = loss / ACCUMULATION_STEPS

            scaler.scale(scaled_loss).backward()
            total_loss += loss.item()

            if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        return total_loss / len(train_loader)


    @torch.no_grad()
    def validate_training_thresholds() -> dict[str, float]:
        model.eval()
        totals = {
            threshold: [0, 0, 0, 0] for threshold in TRAIN_THRESHOLDS
        }
        loss_total = 0.0

        for images, masks, _ in tqdm(val_loader, desc="Validation", leave=False):
            images = move_images(images)
            masks = masks.to(DEVICE, non_blocking=True)
            with autocast_context():
                logits = model(images)
                loss_total += criterion(logits, masks).item()
            probabilities = torch.sigmoid(logits)
            for threshold in TRAIN_THRESHOLDS:
                totals[threshold] = add_values(
                    totals[threshold], confusion(probabilities, masks, threshold)
                )

        rows = []
        for threshold, values in totals.items():
            row = metrics(values)
            row["threshold"] = float(threshold)
            row["loss"] = loss_total / len(val_loader)
            rows.append(row)
        return max(rows, key=lambda item: item["dice"])


    def save_progress_csv(rows):
        if not rows:
            return
        with open(OUT_DIR / "training_history.csv", "w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


    def load_training_checkpoint(path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=DEVICE)


    history: list[dict] = []
    best_dice = -1.0
    best_epoch = 0
    start_epoch = 1
    if LAST_CHECKPOINT_PATH.exists():
        last = load_training_checkpoint(LAST_CHECKPOINT_PATH)
        if last.get("configuration") != configuration:
            raise RuntimeError("Last checkpoint configuration does not match this run.")
        if not CHECKPOINT_PATH.exists():
            raise RuntimeError("The last checkpoint exists but the best checkpoint is missing.")
        model.load_state_dict(last["model"], strict=True)
        optimizer.load_state_dict(last["optimizer"])
        scheduler.load_state_dict(last["scheduler"])
        scaler.load_state_dict(last["scaler"])
        history = last["history"]
        best_dice, best_epoch = last["best_dice"], last["best_epoch"]
        start_epoch = last["epoch"] + 1
        random.setstate(last["python_rng"])
        np.random.set_state(last["numpy_rng"])
        torch.set_rng_state(last["torch_rng"].cpu())
        torch.cuda.set_rng_state_all([state.cpu() for state in last["cuda_rng"]])
        if last["epoch"] - best_epoch >= PATIENCE:
            start_epoch = MAX_EPOCHS + 1
        print(f"Seed {SEED}: resumed after completed epoch {last['epoch']}.")
        del last

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        train_loss = train_epoch()
        validation = validate_training_thresholds()
        scheduler.step(validation["dice"])
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        learning_rate = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": validation["loss"],
            "val_dice": validation["dice"],
            "val_iou": validation["iou"],
            "val_precision": validation["precision"],
            "val_recall": validation["recall"],
            "val_accuracy": validation["accuracy"],
            "val_threshold": validation["threshold"],
            "learning_rate": learning_rate,
            "minutes": elapsed_minutes,
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d} | train {train_loss:.4f} | "
            f"val {validation['loss']:.4f} | Dice {validation['dice']:.4f} | "
            f"IoU {validation['iou']:.4f} | P {validation['precision']:.4f} | "
            f"R {validation['recall']:.4f} | th {validation['threshold']:.2f} | "
            f"lr {learning_rate:.2e} | {elapsed_minutes:.1f}m"
        )

        if validation["dice"] > best_dice:
            best_dice = validation["dice"]
            best_epoch = epoch
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "best_non_tta_val_dice": best_dice,
                    "training_val_threshold": validation["threshold"],
                    "architecture": "UNetPlusPlusBN",
                    "normalization": NORMALIZATION,
                    "in_channels": 6,
                    "base_channels": BASE_CHANNELS,
                    "parameter_count": PARAMETER_COUNT,
                    "input_channel_order": INPUT_CHANNEL_ORDER,
                    "target_size": TARGET_SIZE,
                    "mean": MEAN.tolist(),
                    "std": STD.tolist(),
                    "seed": SEED,
                    "no_tta_primary": True,
                },
                CHECKPOINT_PATH,
            )
            print("Saved:", CHECKPOINT_PATH)

        save_progress_csv(history)
        temporary_checkpoint = LAST_CHECKPOINT_PATH.with_suffix(".tmp")
        torch.save({
            "configuration": configuration, "epoch": epoch,
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
            "history": history, "best_dice": best_dice, "best_epoch": best_epoch,
            "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
        }, temporary_checkpoint)
        temporary_checkpoint.replace(LAST_CHECKPOINT_PATH)
        if epoch - best_epoch >= PATIENCE:
            print(f"Early stopping after {PATIENCE} non-improving epochs.")
            break


    def save_csv(path: Path, rows: list[dict]) -> None:
        if not rows:
            return
        with open(path, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)


    save_csv(OUT_DIR / "training_history.csv", history)


    # =============================================================================
    # 7. FINE VALIDATION THRESHOLD SEARCH AND FINAL NO-TTA TEST
    # =============================================================================

    def safe_load(path: Path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=DEVICE)


    checkpoint = safe_load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()


    @torch.no_grad()
    def collect_no_tta_records(loader: DataLoader, description: str) -> list[dict]:
        records = []
        for images, masks, stems in tqdm(loader, desc=description):
            images = move_images(images)
            with autocast_context():
                probabilities = torch.sigmoid(model(images))
            records.append(
                {
                    "stem": stems[0],
                    "mask": masks[0, 0].numpy().astype(np.float32),
                    "probability": probabilities[0, 0]
                    .float()
                    .cpu()
                    .numpy()
                    .astype(np.float32),
                }
            )
        return records


    def threshold_search(
        records: list[dict], thresholds: list[float]
    ) -> tuple[dict, list[dict]]:
        rows = []
        for threshold in thresholds:
            total = [0, 0, 0, 0]
            for record in records:
                total = add_values(
                    total,
                    confusion(record["probability"], record["mask"], threshold),
                )
            row = metrics(total)
            row["threshold"] = float(threshold)
            rows.append(row)
        return max(rows, key=lambda item: item["dice"]), rows


    def evaluate_records(
        records: list[dict], threshold: float
    ) -> tuple[dict, list[dict]]:
        global_values = [0, 0, 0, 0]
        per_image = []
        for record in records:
            values = confusion(
                record["probability"], record["mask"], threshold
            )
            global_values = add_values(global_values, values)
            row = metrics(values)
            row.update(
                {
                    "stem": record["stem"],
                    "threshold": threshold,
                    "tp": values[0],
                    "fp": values[1],
                    "fn": values[2],
                    "tn": values[3],
                }
            )
            per_image.append(row)

        result = metrics(global_values)
        result.update(
            {
                "threshold": threshold,
                "tp": global_values[0],
                "fp": global_values[1],
                "fn": global_values[2],
                "tn": global_values[3],
            }
        )
        return result, per_image


    # Validation predictions and fine threshold selection happen before any test
    # prediction is generated.
    validation_records = collect_no_tta_records(
        val_loader, "Final validation predictions (no TTA)"
    )
    validation_best, validation_threshold_rows = threshold_search(
        validation_records, FINAL_THRESHOLDS
    )
    selected_threshold = validation_best["threshold"]

    save_csv(
        OUT_DIR / "validation_threshold_search_no_tta.csv",
        validation_threshold_rows,
    )
    save_csv(
        OUT_DIR / "selected_validation_metrics_no_tta.csv",
        [validation_best],
    )

    print("=" * 80)
    print("VALIDATION-SELECTED NO-TTA THRESHOLD:", f"{selected_threshold:.2f}")
    print(json.dumps(validation_best, indent=2))

    # Test set is evaluated exactly once with the frozen validation threshold.
    test_records = collect_no_tta_records(
        test_loader, "Final test predictions (no TTA)"
    )
    test_result, per_image_test = evaluate_records(
        test_records, selected_threshold
    )

    save_csv(
        OUT_DIR / "primary_no_tta_test_metrics.csv",
        [test_result],
    )
    save_csv(
        OUT_DIR / "per_image_test_metrics_no_tta.csv",
        per_image_test,
    )

    run_summary = {
        "model": "U-Net++ (custom convolutional encoder, BatchNorm)",
        "source_sha256": PROGRAM_SHA256,
        "encoder_initialization": "from scratch",
        "model_source": "graindiscoloration (2).ipynb, cell 73 (zero-based)",
        "training_protocol_source": "shared controlled protocol in notebook cell 45",
        "dataset_root": str(BASE_DIR),
        "seed": SEED,
        "split_sizes": {
            "train": len(train_names),
            "validation": len(val_names),
            "test": len(test_names),
        },
        "input_channels": INPUT_CHANNEL_ORDER,
        "include_ndre": False,
        "target_size_width_height": list(TARGET_SIZE),
        "normalization_layer": NORMALIZATION,
        "normalization_mean": MEAN.tolist(),
        "normalization_std": STD.tolist(),
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "epochs_requested": MAX_EPOCHS,
        "best_epoch": checkpoint["epoch"],
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=4,min_lr=1e-6)",
        "loss": "0.35 BCE + 0.25 focal(gamma=2) + 0.40 Dice",
        "parameter_count": PARAMETER_COUNT,
        "checkpoint_selected_by": "highest coarse-grid validation Dice",
        "threshold_selected_by": "highest fine-grid validation Dice",
        "selected_threshold": selected_threshold,
        "test_time_augmentation": "none",
        "validation_metrics": validation_best,
        "primary_no_tta_test_metrics": test_result,
    }

    temporary_summary = OUT_DIR / "run_summary.json.tmp"
    temporary_summary.write_text(json.dumps(run_summary, indent=2))
    temporary_summary.replace(OUT_DIR / "run_summary.json")

    print("=" * 80)
    print("FINAL PRIMARY TEST RESULT — NO TTA")
    print("=" * 80)
    print(f"Dice      : {test_result['dice']:.4f}")
    print(f"IoU       : {test_result['iou']:.4f}")
    print(f"Precision : {test_result['precision']:.4f}")
    print(f"Recall    : {test_result['recall']:.4f}")
    print(f"Threshold : {test_result['threshold']:.2f}")
    print(f"Parameters: {PARAMETER_COUNT / 1e6:.3f} M")
    print("Best epoch:", checkpoint["epoch"])

    archive_base = MASTER_OUT_DIR / f"gd_unetpp_bn_6ch_seed{SEED}_results"
    archive_path = Path(
        shutil.make_archive(str(archive_base), "zip", root_dir=OUT_DIR)
    )
    print("Results folder:", OUT_DIR)
    print("Results ZIP   :", archive_path)

    if SHOW_DOWNLOAD_LINK:
        try:
            from google.colab import files

            files.download(str(archive_path))
        except (ImportError, ModuleNotFoundError):
            pass

    print("=" * 80)
    print(f"U-NET++ SEED {SEED} FINISHED")
    print("=" * 80)
    return run_summary


def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def main():
    global PROGRAM_SHA256
    if RUN_SEEDS != [123, 2026]:
        raise ValueError("This runner is restricted to seeds 123 and 2026; seed 42 is not rerun.")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator in Kaggle/Colab, then run this cell.")
    # Identifies the exact generated architecture and controlled protocol.
    PROGRAM_SHA256 = SOURCE_CONFIGURATION_SHA256
    print("U-NET++ | SIX CHANNELS | SEEDS 123 AND 2026 ONLY")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Output:", MASTER_OUT_DIR)
    (MASTER_OUT_DIR / "seed42_manuscript_reference.json").write_text(
        json.dumps(SEED42_REFERENCE, indent=2)
    )
    new_rows = []
    for seed in RUN_SEEDS:
        summary = run_one(seed)
        metrics = summary["primary_no_tta_test_metrics"]
        row = {"model": "U-Net++", "seed": seed,
               **{key: metrics[key] for key in ["dice", "iou", "precision", "recall", "accuracy", "threshold"]},
               "best_epoch": summary["best_epoch"],
               "parameter_count": summary["parameter_count"]}
        new_rows.append(row)
        write_csv(MASTER_OUT_DIR / "new_seed_results.csv", new_rows)
        del summary
        gc.collect()
        torch.cuda.empty_cache()

    print("\nFINAL RESULTS — ACTUAL NEW RUNS")
    for row in new_rows:
        print(f"Seed {row['seed']} | Dice {row['dice']:.6f} | IoU {row['iou']:.6f} | "
              f"P {row['precision']:.6f} | R {row['recall']:.6f} | th {row['threshold']:.2f}")

    combined = [{"seed": 42, **{key: SEED42_REFERENCE[key] for key in ["dice", "iou", "precision", "recall"]},
                 "provenance": SEED42_REFERENCE["provenance"]}]
    combined.extend({"seed": r["seed"], **{key: r[key] for key in ["dice", "iou", "precision", "recall"]},
                     "provenance": "new completed run"} for r in new_rows)
    write_csv(MASTER_OUT_DIR / "three_seed_values_with_seed42_reference.csv", combined)
    summary_rows = []
    for key in ["dice", "iou", "precision", "recall"]:
        values = np.asarray([r[key] for r in combined], dtype=np.float64)
        row = {"metric": key, "n_seeds": 3, "mean": float(values.mean()),
               "sample_sd": float(values.std(ddof=1)),
               "seed42_source": "existing manuscript rounded values"}
        summary_rows.append(row)
    write_csv(MASTER_OUT_DIR / "three_seed_mean_sd_with_seed42_reference.csv", summary_rows)
    print("\nTHREE-SEED SUMMARY — includes the existing rounded seed-42 manuscript values")
    for row in summary_rows:
        print(f"{row['metric']:10s}: {row['mean']:.4f} +/- {row['sample_sd']:.4f}")
    archive = str(MASTER_OUT_DIR) + "_complete.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(MASTER_OUT_DIR.rglob("*")):
            if path.is_file() and path.suffix not in {".zip", ".tmp"}:
                bundle.write(path, path.relative_to(MASTER_OUT_DIR))
    print("\nFinal ZIP:", archive)
    try:
        from IPython.display import display, FileLink
        display(FileLink(archive))
    except ImportError:
        pass


SOURCE_CONFIGURATION_SHA256 = 'bd1d9a9f5b7905180df0161188608ce6502c76903749cbb7f7ee988d167410e9'

if __name__ == "__main__":
    main()
